# 02 - Data Preprocessing & Cleaning
## CSE-CIC-IDS2018

**Input:** `file_100.csv`, `file_75.csv`, `file_50.csv`, `file_25.csv`

**Output:** `cleaned_100.pkl`, `cleaned_75.pkl`, `cleaned_50.pkl`, `cleaned_25.pkl`

In [ ]:
import pandas as pd
import numpy as np
import os, gc, pickle
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Kolom non-fitur yang harus dibuang
REMOVE_COLS = ['Timestamp', 'Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Protocol', 'Label']

INPUT_FILES = ['file_100.csv', 'file_75.csv', 'file_50.csv', 'file_25.csv']
OUTPUT_FILES = ['cleaned_100.pkl', 'cleaned_75.pkl', 'cleaned_50.pkl', 'cleaned_25.pkl']

print(f'Data dir: {DATA_DIR}')
print(f'Columns to remove: {REMOVE_COLS}')
print('✓ Setup complete')

## 1. Proses file_100.csv (Referensi)

In [ ]:
# Load file_100
print('Loading file_100.csv...')
df = pd.read_csv(os.path.join(DATA_DIR, 'file_100.csv'), low_memory=False)
df.columns = df.columns.str.strip()
print(f'  Loaded: {df.shape}')

# Remove embedded header rows (where Label == 'Label')
mask = df['Label'].astype(str).str.strip().str.lower() != 'label'
removed = (~mask).sum()
df = df[mask].reset_index(drop=True)
print(f'  Removed {removed} header rows → {df.shape}')

# Separate label
y_raw = df['Label'].astype(str).str.strip()

# Drop non-feature columns
cols_to_drop = [c for c in REMOVE_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f'  Dropped {len(cols_to_drop)} columns → {df.shape}')
print(f'  Dropped: {cols_to_drop}')

In [ ]:
# Convert to numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Replace inf with NaN, then fill NaN with median
df = df.replace([np.inf, -np.inf], np.nan)
nan_count = df.isna().sum().sum()
print(f'  NaN/Inf values: {nan_count}')
df = df.fillna(df.median())

# Drop zero-variance columns
variances = df.var()
zero_var_cols = variances[variances == 0].index.tolist()
if zero_var_cols:
    df = df.drop(columns=zero_var_cols)
print(f'  Zero-variance removed ({len(zero_var_cols)}): {zero_var_cols}')
print(f'  Final features: {df.shape}')

# Save feature names
feature_names = df.columns.tolist()
print(f'  Feature count: {len(feature_names)}')

In [ ]:
# StandardScaler (using .values to avoid dtype issues)
scaler = StandardScaler()
X = scaler.fit_transform(df.values)
print(f'  Scaled X: {X.shape} | mean≈{X.mean():.6f} | std≈{X.std():.6f}')

# LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_raw)
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(f'  Labels encoded: {len(label_mapping)} classes')
print(f'  Classes: {list(le.classes_)}')

# Save
output = {
    'X': X, 'y': y,
    'feature_names': feature_names,
    'label_mapping': label_mapping,
    'scaler': scaler,
    'label_encoder': le,
    'zero_var_cols': zero_var_cols
}
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'wb') as f:
    pickle.dump(output, f)

print(f'\n✓ Saved cleaned_100.pkl (X={X.shape}, y={y.shape})')
del df; gc.collect()

## 2. Proses file_75, file_50, file_25 (menggunakan referensi dari file_100)

In [ ]:
for input_file, output_file in zip(INPUT_FILES[1:], OUTPUT_FILES[1:]):
    print(f'\n{"="*60}')
    print(f'Processing: {input_file}')
    print(f'{"="*60}')
    
    filepath = os.path.join(DATA_DIR, input_file)
    if not os.path.exists(filepath):
        print(f'  NOT FOUND — skipping')
        continue
    
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f'  Loaded: {df.shape}')
    
    # Remove header rows
    mask = df['Label'].astype(str).str.strip().str.lower() != 'label'
    df = df[mask].reset_index(drop=True)
    
    # Separate label
    y_raw = df['Label'].astype(str).str.strip()
    
    # Drop non-feature columns
    cols_to_drop = [c for c in REMOVE_COLS if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    
    # Convert to numeric + handle inf/nan
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(df.median())
    
    # Drop same zero-variance columns as file_100
    cols_exist = [c for c in zero_var_cols if c in df.columns]
    df = df.drop(columns=cols_exist)
    
    # Keep only same features as file_100 (in same order)
    missing_feats = [f for f in feature_names if f not in df.columns]
    if missing_feats:
        print(f'  WARNING: missing features: {missing_feats}')
        for f in missing_feats:
            df[f] = 0
    df = df[feature_names]
    
    # Scale using same scaler
    X = scaler.transform(df.values)
    
    # Encode labels (handle unseen)
    known = set(le.classes_)
    y_clean = y_raw.apply(lambda x: x if x in known else 'UNKNOWN')
    if 'UNKNOWN' in y_clean.values and 'UNKNOWN' not in known:
        le.classes_ = np.append(le.classes_, 'UNKNOWN')
    y = le.transform(y_clean)
    
    # Save
    out = {
        'X': X, 'y': y,
        'feature_names': feature_names,
        'label_mapping': dict(zip(le.classes_, range(len(le.classes_)))),
        'scaler': scaler,
        'label_encoder': le
    }
    with open(os.path.join(DATA_DIR, output_file), 'wb') as f:
        pickle.dump(out, f)
    
    print(f'  ✓ Saved {output_file} (X={X.shape}, y={y.shape})')
    del df; gc.collect()

print(f'\n{"="*60}')
print('ALL FILES PROCESSED!')
print(f'{"="*60}')

## 3. Sanity Check

In [ ]:
# Verify cleaned_100
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    check = pickle.load(f)

print('Sanity check cleaned_100.pkl:')
print(f'  X: {check["X"].shape} | mean={check["X"].mean():.4f} | std={check["X"].std():.4f}')
print(f'  y: {check["y"].shape} | unique={np.unique(check["y"])}')
print(f'  NaN: {np.isnan(check["X"]).sum()} | Inf: {np.isinf(check["X"]).sum()}')
print(f'  Features: {len(check["feature_names"])}')
print(f'  Label mapping: {check["label_mapping"]}')
print(f'\n✓ All checks passed!')